# Topic G — Practice-Final P4: MC Delta Bias-Variance Trade-off

*FINM 32000 Numerical Methods — Final-Exam Study Notes*

---

**Prerequisites:** [Topic E — MC Foundations](Topic-E_L5_MonteCarlo_Foundations.ipynb) (MC estimator, SE, bumping for delta).

**What this notebook covers**
1. The MC delta estimator and where it goes wrong
2. **Bias** from forward differences — what it scales like
3. **Variance** of a bumped MC delta — what it scales like
4. **Mean squared error (MSE)** combines them
5. **Optimal $\varepsilon$** that minimizes MSE — exam derivation
6. **Common random numbers** save you — and what happens without them
7. Numerical demo of the trade-off
8. Cheat sheet

**Exam map**

| Source | What it tests |
|---|---|
| Practice-Final P4(a) | Write MSE in terms of $a, b, M, \varepsilon$ |
| Practice-Final P4(b) | Solve for optimal $\varepsilon$ |
| HW5 P1(d) | Computing delta by bumping |

**Why this matters:** This is the **only purely analytic** MC problem on the practice final. The math is short but the bookkeeping is delicate; do it once and it sticks.


In [1]:
import numpy as np
from scipy.stats import norm
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(precision=6, suppress=True)

def bs_call(S, K, T, r, sigma):
    if T <= 0: return max(S - K, 0.0)
    std = sigma * np.sqrt(T)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / std
    d2 = d1 - std
    return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)

def bs_delta(S, K, T, r, sigma):
    std = sigma * np.sqrt(T)
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / std
    return norm.cdf(d1)


## 1. The setup

We want the time-0 delta of a contract:
$$\Delta := \frac{\partial C}{\partial S}(S_0).$$

**MC estimator** with bumping $\varepsilon > 0$ and $M$ paths each side:
$$\widehat\Delta_M := \frac{\widehat C_M(S_0 + \varepsilon) - \widehat C_M(S_0)}{\varepsilon},$$
where $\widehat C_M(s)$ is the standard MC price estimator with initial spot $s$.

This is the standard recipe used in HW5 P1(d) and across the course.

### Two sources of error

1. **Bias** from approximating a derivative by a finite difference.
2. **Variance** from MC noise in $\widehat C_M$.

The exam question (Practice-Final P4) just plugs in the **scaling laws** for each — we are told the constants $a$ and $b$ exist; we are asked for the **form** of MSE and the optimal $\varepsilon$.


## 2. Bias

### Taylor expansion

For smooth $C$:
$$\frac{C(S_0 + \varepsilon) - C(S_0)}{\varepsilon} = C'(S_0) + \tfrac12 C''(S_0)\,\varepsilon + O(\varepsilon^2).$$

So the **forward difference** has bias
$$\frac{C(S_0+\varepsilon) - C(S_0)}{\varepsilon} - C'(S_0) = b\,\varepsilon, \qquad b := \tfrac12 C''(S_0) = \tfrac12 \Gamma.$$

**Plain English:** Bias is proportional to $\varepsilon$. Smaller $\varepsilon$ → less bias.

### Practice-Final P4 wording

> "Assume that the bias is exactly $b\varepsilon$ for some constant $b$."

We take this as given. (For a central difference, the bias would be $O(\varepsilon^2)$ — better — but the problem fixes the forward-difference setup.)


## 3. Variance

### What scaling do we expect?

The Practice-Final P4 problem **gives** us:
$$\mathrm{Var}(\widehat\Delta_M) = \frac{a}{M\varepsilon}.$$

This **does not** match independent-$Z$ MC. Let me explain both cases.

### Case A — common random numbers (good)

Use the **same** $Z$ draws for $\widehat C_M(S_0+\varepsilon)$ and $\widehat C_M(S_0)$. Then the per-path delta is
$$D_m = \frac{Y_m(S_0+\varepsilon) - Y_m(S_0)}{\varepsilon}.$$

For a GBM call payoff $Y_m(s) = e^{-rT}(s\,e^{(r-\sigma^2/2)T + \sigma\sqrt T Z_m} - K)^+$, **most paths give the same $D_m$** because the indicator $\{S_T > K\}$ doesn't flip. Variance per path is finite even as $\varepsilon \to 0$.

So in this best case, $\mathrm{Var}(\widehat\Delta_M) = O(1/M)$, *not* $O(1/(M\varepsilon))$.

### Case B — independent $Z$ (the case in Practice-Final P4)

Use **separate** random draws for each side. Then
$$\mathrm{Var}(\widehat\Delta_M) = \frac{\mathrm{Var}(\widehat C_M(S_0+\varepsilon)) + \mathrm{Var}(\widehat C_M(S_0))}{\varepsilon^2} \approx \frac{2\sigma_C^2/M}{\varepsilon^2}.$$

For small $\varepsilon$ this is $\sim 1/(M\varepsilon^2)$, which is **worse** than the problem's stated $1/(M\varepsilon)$.

### Reading the problem

Practice-Final P4 says the variance is $a/(M\varepsilon)$ "for some constant $a$." That intermediate scaling sits between the two extremes — perhaps imagining **partial reuse** of random numbers, or a specific contract where the two MC estimates have correlation that scales linearly in $\varepsilon$.

**For the exam, accept the given $a/(M\varepsilon)$ at face value.** The whole problem is about how the optimal $\varepsilon$ depends on the powers of $\varepsilon$ in bias and variance.


## 4. Mean squared error — Practice-Final P4(a)

### MSE = variance + bias squared

For **any** estimator $\widehat D$ of nonrandom target $D$:
$$\mathbb E[(\widehat D - D)^2] = \mathrm{Var}(\widehat D) + (\mathbb E[\widehat D] - D)^2.$$

(Proof: $\widehat D - D = (\widehat D - \mathbb E\widehat D) + (\mathbb E\widehat D - D)$; expand the square; the cross term vanishes.)

### Plug in

$$\boxed{\mathrm{MSE}(\widehat\Delta_M) = \mathrm{Var}(\widehat\Delta_M) + \mathrm{Bias}^2 = \frac{a}{M\varepsilon} + b^2\varepsilon^2.}$$

This is the answer to Practice-Final P4(a). It is **one line**.

### Reading the trade-off

- **$\varepsilon$ too small** → bias is tiny but variance term $a/(M\varepsilon)$ explodes.
- **$\varepsilon$ too large** → variance shrinks but bias term $b^2\varepsilon^2$ explodes.
- There is a **sweet spot** that minimizes the sum.


## 5. Optimal $\varepsilon$ — Practice-Final P4(b)

### Differentiate

Set $\dfrac{d}{d\varepsilon}\Bigl[\dfrac{a}{M\varepsilon} + b^2\varepsilon^2\Bigr] = 0$:
$$-\frac{a}{M\varepsilon^2} + 2b^2\varepsilon = 0.$$

### Solve

$$2b^2\varepsilon^3 = \frac{a}{M}, \qquad \varepsilon^3 = \frac{a}{2M b^2}.$$

$$\boxed{\varepsilon^* = \left(\frac{a}{2Mb^2}\right)^{1/3}.}$$

### Plug back to get optimal MSE

At $\varepsilon^*$:
$$\frac{a}{M\varepsilon^*} = \frac{a}{M}\left(\frac{2Mb^2}{a}\right)^{1/3} = (2ab)^{2/3} M^{-2/3},$$
$$b^2(\varepsilon^*)^2 = b^2\left(\frac{a}{2Mb^2}\right)^{2/3} = \frac{(2ab)^{2/3}}{4}\,M^{-2/3}\cdot 2 = \text{(same order)}.$$

So $\mathrm{MSE}^* = O(M^{-2/3})$ → $\widehat\Delta - \Delta = O(M^{-1/3})$.

**Plain English:** **MC delta converges slower than MC prices** — only $M^{-1/3}$ instead of $M^{-1/2}$. Reducing the gap requires $M^3$ more paths to halve the delta error.

### Practical implication

Even with the optimal $\varepsilon$, MC delta is **inherently noisy**. Two cures:
1. **Common random numbers** (Case A above) — restores $M^{-1/2}$.
2. **Pathwise** or **likelihood-ratio** methods (not on this final).


In [2]:
# Numerical demonstration of the bias-variance trade-off
np.random.seed(0)
S0, K, T, r, sigma = 100, 100, 1.0, 0.05, 0.20
true_delta = bs_delta(S0, K, T, r, sigma)
M = 50_000

def mc_call(S, M, rng):
    Z = rng.standard_normal(M)
    ST = S * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
    return np.exp(-r*T) * np.maximum(ST - K, 0.0)

epsilons = np.array([0.001, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0, 5.0])
n_trials = 50

# Case B: independent Z each side -> heavy variance for small epsilon
results_indep = np.zeros((len(epsilons), n_trials))
for i, eps in enumerate(epsilons):
    for t in range(n_trials):
        rng1 = np.random.default_rng(t * 2)
        rng2 = np.random.default_rng(t * 2 + 1)
        c0 = mc_call(S0, M, rng1).mean()
        c1 = mc_call(S0 + eps, M, rng2).mean()
        results_indep[i, t] = (c1 - c0) / eps

# Case A: common random numbers
results_common = np.zeros((len(epsilons), n_trials))
for i, eps in enumerate(epsilons):
    for t in range(n_trials):
        rng = np.random.default_rng(t)
        Z = rng.standard_normal(M)
        S_T0 = S0 * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
        S_Te = (S0 + eps) * np.exp((r - 0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
        c0 = np.exp(-r*T)*np.maximum(S_T0 - K, 0.0).mean()
        c1 = np.exp(-r*T)*np.maximum(S_Te - K, 0.0).mean()
        results_common[i, t] = (c1 - c0) / eps

bias_indep   = results_indep.mean(axis=1)   - true_delta
sd_indep     = results_indep.std(axis=1, ddof=1)
mse_indep    = bias_indep**2 + sd_indep**2

bias_common  = results_common.mean(axis=1)  - true_delta
sd_common    = results_common.std(axis=1, ddof=1)
mse_common   = bias_common**2 + sd_common**2

print(f"True delta: {true_delta:.4f}\n")
print(f"{'epsilon':>8} | {'Case A (CRN)':>30} | {'Case B (indep)':>30}")
print(f"{'':>8} | {'bias':>9} {'sd':>9} {'sqrt MSE':>9} | {'bias':>9} {'sd':>9} {'sqrt MSE':>9}")
print("-" * 80)
for i, eps in enumerate(epsilons):
    print(f"{eps:>8.3f} | {bias_common[i]:>9.4f} {sd_common[i]:>9.4f} "
          f"{np.sqrt(mse_common[i]):>9.4f} | {bias_indep[i]:>9.4f} {sd_indep[i]:>9.4f} "
          f"{np.sqrt(mse_indep[i]):>9.4f}")
print()
print("Notes:")
print(" - Common random numbers (A): sd is tiny and stable -> bias dominates")
print(" - Independent draws (B):     sd explodes as epsilon shrinks -> need to balance")


True delta: 0.6368

 epsilon |                   Case A (CRN) |                 Case B (indep)
         |      bias        sd  sqrt MSE |      bias        sd  sqrt MSE
--------------------------------------------------------------------------------
   0.001 |    0.0001    0.0024    0.0024 |  -17.9212  106.0000  107.5043
   0.010 |    0.0002    0.0024    0.0024 |   -1.7918   10.6014   10.7518
   0.050 |    0.0005    0.0024    0.0025 |   -0.3577    2.1215    2.1515
   0.100 |    0.0010    0.0024    0.0026 |   -0.1780    1.0615    1.0764
   0.500 |    0.0048    0.0024    0.0053 |   -0.0310    0.2135    0.2158
   1.000 |    0.0094    0.0024    0.0097 |   -0.0085    0.1075    0.1078
   2.000 |    0.0185    0.0023    0.0186 |    0.0096    0.0545    0.0554
   5.000 |    0.0446    0.0023    0.0447 |    0.0411    0.0228    0.0470

Notes:
 - Common random numbers (A): sd is tiny and stable -> bias dominates
 - Independent draws (B):     sd explodes as epsilon shrinks -> need to balance


## 6. Common random numbers — the key real-world fix

### Why it works

When the **same** $Z_m$ drives both $\widehat C_M(S_0+\varepsilon)$ and $\widehat C_M(S_0)$, the per-path delta
$$D_m = \frac{Y_m(S_0 + \varepsilon) - Y_m(S_0)}{\varepsilon}$$
is the finite-difference approximation of $\partial Y_m/\partial S_0$ along the **same path realization**. For continuous payoffs (e.g., calls, puts) this is bounded and well-behaved as $\varepsilon \to 0$ — the **variance per path stays $O(1)$**.

### When CRN can fail

For **discontinuous payoffs** (digital options, knockouts at exact barriers), even CRN gives explosive variance because $\partial Y/\partial S_0$ does not exist on a set of positive measure. For such cases, smoothing or likelihood-ratio methods are needed.

### Practical exam tip

If you see a problem about computing $\partial C/\partial S$ by MC bumping, **always use the same $Z$ matrix** for both estimators. The HW5 P1 code does exactly this.


## 7. Worked Practice-Final P4 — write the solutions

### (a) Mean squared error

$$\mathrm{MSE}(\widehat\Delta_M) = \mathrm{Var}(\widehat\Delta_M) + (\text{bias})^2 = \frac{a}{M\varepsilon} + b^2\varepsilon^2.$$

(Using the hint: $\mathrm{Var}(\widehat\Delta_M) = \mathrm{Var}(\widehat\Delta_M - \Delta)$ since $\Delta$ is nonrandom.)

### (b) Optimal $\varepsilon$

Differentiate MSE in $\varepsilon$ and set to zero:
$$-\frac{a}{M\varepsilon^2} + 2b^2\varepsilon = 0 \;\Longrightarrow\; \varepsilon = \left(\frac{a}{2Mb^2}\right)^{1/3}.$$

That's the entire problem. Two lines of algebra after setting up the MSE.


## 8. Common pitfalls

### Pitfall 1 — confusing bias and variance scalings

Memorize: **bias $\propto \varepsilon$, variance $\propto 1/(M\varepsilon)$** in the Practice-Final P4 setup. The minimum of a sum of $\varepsilon^k$ and $\varepsilon^{-j}$ is always at $\varepsilon^{k+j} = $ ratio of coefficients.

### Pitfall 2 — using a quadratic-bias result for a forward difference

A **central** difference $(C(S+\varepsilon) - C(S-\varepsilon))/(2\varepsilon)$ has bias $O(\varepsilon^2)$, not $O(\varepsilon)$. If the problem uses central differences, MSE becomes $a/(M\varepsilon) + b^2\varepsilon^4$ and the optimal $\varepsilon$ is $(a/(4Mb^2))^{1/5}$. Same recipe.

### Pitfall 3 — too-small $\varepsilon$ on a real run

Even in CRN, very small $\varepsilon$ (like $10^{-8}$) hits **floating-point noise**. Practical $\varepsilon$: typically $10^{-3}$ to $10^{-2}$ for stock-price bumping.

### Pitfall 4 — bumping twice with different RNG state

If you naively call `rng.normal()` twice (once at $S_0$, once at $S_0+\varepsilon$), you get **different** $Z$'s. Fix: pre-generate $Z$ once and pass it in, **or** reset the RNG seed.


## 9. Exam-style exercises

**E1.** For a central-difference delta estimator with bias $b\varepsilon^2$ and variance $a/(M\varepsilon)$, find the optimal $\varepsilon$.

**E2.** In the original Practice-Final P4 setup, by what factor does the optimal $\varepsilon$ shrink if you increase $M$ by $1000\times$?

**E3.** A more careful analysis of CRN gives $\mathrm{Var}(\widehat\Delta_M) = a/M$ (independent of $\varepsilon$). What is the optimal $\varepsilon$?

**E4.** Why is MC delta only $O(M^{-1/3})$ in Practice-Final P4 setup?


### Solutions

<details><summary><b>E1</b></summary>

MSE $= a/(M\varepsilon) + b^2\varepsilon^4$. Differentiate: $-a/(M\varepsilon^2) + 4b^2\varepsilon^3 = 0$. So $\varepsilon^5 = a/(4Mb^2)$ and $\varepsilon^* = (a/(4Mb^2))^{1/5}$.
</details>

<details><summary><b>E2</b></summary>

$\varepsilon^* \propto M^{-1/3}$, so $1000\times$ more $M$ shrinks $\varepsilon^*$ by $1000^{1/3} = 10\times$.
</details>

<details><summary><b>E3</b></summary>

MSE $= a/M + b^2\varepsilon^2$. The variance term doesn't depend on $\varepsilon$ at all, so MSE is minimized by taking $\varepsilon \to 0$. In practice, floating-point error becomes the constraint, not bias-variance.
</details>

<details><summary><b>E4</b></summary>

$\mathrm{MSE}^* = O(M^{-2/3})$ from the formula in §5, so $\sqrt{\mathrm{MSE}^*} = O(M^{-1/3})$. Compare to MC price error $O(M^{-1/2})$ — delta is intrinsically harder when you do it naively.
</details>


## 10. Cheat sheet — Topic G

| # | Fact | Where |
|---|---|---|
| 1 | $\widehat\Delta_M = (\widehat C_M(S_0+\varepsilon) - \widehat C_M(S_0))/\varepsilon$ | Forward difference |
| 2 | Bias $= b\varepsilon$, $b = \tfrac12 C''(S_0)$ | Taylor |
| 3 | $\mathrm{Var}(\widehat\Delta) = a/(M\varepsilon)$ | Given in P4 |
| 4 | MSE $= \mathrm{Var} + \mathrm{bias}^2 = a/(M\varepsilon) + b^2\varepsilon^2$ | P4(a) |
| 5 | $\varepsilon^* = (a/(2Mb^2))^{1/3}$ | P4(b) |
| 6 | $\mathrm{MSE}^* = O(M^{-2/3})$ | Convergence rate |
| 7 | Central diff: bias $\propto \varepsilon^2$ | Two-sided is better |
| 8 | **Always use common random numbers** | Practical fix |
| 9 | CRN fails for discontinuous payoffs | Need smoothing |

---

*Next:* **Topic H — Longstaff-Schwartz for American options** (Practice-Final P5 bug spotting + HW7 P1).
